In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets

In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_data = datasets.ImageFolder(root='assign_data/training_set/training_set', transform=transform)
test_data = datasets.ImageFolder(root='assign_data/test_set/test_set', transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [16]:
train_data

Dataset ImageFolder
    Number of datapoints: 8005
    Root location: assign_data/training_set/training_set
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

## BUILD CNN

In [17]:
import torch
print(torch.cuda.is_available())

True


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [19]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64*32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [20]:
import torch.optim as optim

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training the MOdel

In [21]:
for epoch in range(10):
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.6613635551407042
Epoch 2, Loss: 0.5365443568068197
Epoch 3, Loss: 0.4425767354400034
Epoch 4, Loss: 0.3357152684157588
Epoch 5, Loss: 0.21742738741802503
Epoch 6, Loss: 0.1006624656339803
Epoch 7, Loss: 0.03592684110107115
Epoch 8, Loss: 0.01582751509504378
Epoch 9, Loss: 0.02808464195118267
Epoch 10, Loss: 0.005811909808493424


## Evaluation

In [22]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print("Accuracy:", (correct/total)*100)

Accuracy: 74.83934750370736


In [10]:
print(len(train_data))
print(len(test_data))


8005
2023


In [11]:
print(len(test_loader))

64


In [12]:
print(loss)

tensor(0.2855, device='cuda:0', grad_fn=<NllLossBackward0>)


In [13]:
print(outputs[:5])
print(labels[:5])

tensor([[ 1.3676, -1.6586],
        [ 0.0365, -0.1201],
        [-0.4371,  0.7188],
        [-1.2994,  1.1999],
        [ 0.9867, -1.1004]], device='cuda:0')
tensor([1, 1, 1, 1, 1], device='cuda:0')


In [14]:
from collections import Counter

labels_list = [label for _, label in train_data]
print(Counter(labels_list))

Counter({1: 4005, 0: 4000})
